# OMIP Simulation Diagnostics -- Multi-case comparison

Post-processing visualization loosely following Adcroft et al. (2019),
*The GFDL Global Ocean and Sea Ice Model OM4.0*, JAMES.

Define two (or more) cases below; every figure shows them side by side.

In [ ]:
cases = [
    (run_dir = "halfdegree_run", prefix = "halfdegree", label = "Half-degree"),
    (run_dir = "orca_run",       prefix = "orca",       label = "ORCA"),
]

# Averaging window in seconds from simulation start.
# Only snapshots with start_time <= t <= stop_time are used for time means.
# Set start_time = 0 and stop_time = Inf to use the full time series.
start_time = 0
stop_time  = Inf

In [ ]:
using CairoMakie
using Statistics
using Dates
using Oceananigans
using Oceananigans.Grids: znodes, φnodes
using Oceananigans.Fields: interpolate!
using ConservativeRegridding
using NumericalEarth
using NumericalEarth.DataWrangling: Metadatum
using NumericalEarth.DataWrangling.WOA: WOAAnnual

In [ ]:
# ── Helpers ──────────────────────────────────────────────────

function find_first_file(run_dir, prefix, group)
    tag = "$(prefix)_$(group)"
    candidates = filter(f -> startswith(f, tag) && endswith(f, ".jld2") &&
                             !contains(f, "checkpoint"), readdir(run_dir))
    isempty(candidates) && error("No $group files for prefix '$prefix' in $run_dir")
    # Strip _partN suffix so FieldTimeSeries discovers all parts
    filename = first(sort(candidates))
    basename_no_part = replace(filename, r"_part\d+" => "")
    return joinpath(run_dir, basename_no_part)
end

"""Return indices into `fts.times` that fall within [start_time, stop_time]."""
function in_window(fts; start_time = 0, stop_time = Inf)
    return findall(t -> start_time <= t <= stop_time, fts.times)
end

function compute_time_mean(fts; start_time = 0, stop_time = Inf)
    idx = in_window(fts; start_time, stop_time)
    isempty(idx) && error("No snapshots in [$start_time, $stop_time]")
    avg = zeros(size(Array(interior(fts[first(idx)]))))
    for n in idx
        avg .+= Array(interior(fts[n]))
    end
    return avg ./ length(idx)
end

function compute_monthly_mean(fts, target_months;
                              start_time = 0, stop_time = Inf,
                              reference_date = DateTime(1958, 1, 1))
    dates = [reference_date + Second(round(Int, t)) for t in fts.times]
    idx   = findall(i -> month(dates[i]) in target_months &&
                         start_time <= fts.times[i] <= stop_time,
                    eachindex(dates))
    isempty(idx) && return nothing
    avg = zeros(size(Array(interior(fts[first(idx)]))))
    for n in idx
        avg .+= Array(interior(fts[n]))
    end
    return avg ./ length(idx)
end

function build_land_mask(grid)
    if grid isa ImmersedBoundaryGrid
        bh = Array(interior(grid.immersed_boundary.bottom_height, :, :, 1))
        return bh .>= 0
    else
        return falses(size(grid, 1), size(grid, 2))
    end
end

function build_ocean_mask_3d(grid)
    Nx, Ny, Nz = size(grid)
    mask = ones(Nx, Ny, Nz)
    if grid isa ImmersedBoundaryGrid
        bh = Array(interior(grid.immersed_boundary.bottom_height, :, :, 1))
        zc = znodes(grid, Center())
        for k in 1:Nz, j in 1:Ny, i in 1:Nx
            zc[k] < bh[i, j] && (mask[i, j, k] = 0.0)
        end
    end
    return mask
end

mask_land!(f, land) = (f[land] .= NaN; f)

function panel!(fig, pos, data;
                title="", colormap=:thermal,
                colorrange=nothing, label="",
                nan_color=:lightgray)
    ax = Axis(fig[pos...]; title)
    kw = isnothing(colorrange) ? (;) : (; colorrange)
    hm = heatmap!(ax, data; colormap, nan_color, kw...)
    Colorbar(fig[pos[1], pos[2]+1], hm; label)
    return ax
end

case_colors = [:firebrick, :royalblue, :seagreen, :darkorange]

## Load surface diagnostics for all cases

In [ ]:
function load_surface_case(run_dir, prefix; start_time = 0, stop_time = Inf)
    surface_file = find_first_file(run_dir, prefix, "surface")
    @info "  surface: $surface_file"

    tos     = FieldTimeSeries(surface_file, "tos";    backend = OnDisk())
    sos     = FieldTimeSeries(surface_file, "sos";    backend = OnDisk())
    zos     = FieldTimeSeries(surface_file, "zos";    backend = OnDisk())
    mld_fts = FieldTimeSeries(surface_file, "mlotst"; backend = OnDisk())
    hfds    = FieldTimeSeries(surface_file, "hfds";   backend = OnDisk())
    wfo     = FieldTimeSeries(surface_file, "wfo";    backend = OnDisk())
    sic     = FieldTimeSeries(surface_file, "siconc"; backend = OnDisk())
    zossq   = FieldTimeSeries(surface_file, "zossq";  backend = OnDisk())

    grid = tos.grid
    Nx, Ny, Nz = size(grid)
    land = build_land_mask(grid)

    @info "  averaging window: [$(start_time / (365.25*86400)), $(stop_time / (365.25*86400))] years"

    SST = dropdims(compute_time_mean(tos;  start_time, stop_time);  dims=3)
    SSS = dropdims(compute_time_mean(sos;  start_time, stop_time);  dims=3)
    SSH = dropdims(compute_time_mean(zos;  start_time, stop_time);  dims=3)
    HF  = dropdims(compute_time_mean(hfds; start_time, stop_time);  dims=3)
    FW  = dropdims(compute_time_mean(wfo;  start_time, stop_time);  dims=3)
    SIC_mean = dropdims(compute_time_mean(sic; start_time, stop_time); dims=3)

    # SSH variance
    SSH_sq = dropdims(compute_time_mean(zossq; start_time, stop_time); dims=3)
    SSH_var = SSH_sq .- SSH .^ 2

    # MLD min / max over annual cycle
    MLD_monthly = [compute_monthly_mean(mld_fts, [m]; start_time, stop_time) for m in 1:12]
    avail = findall(!isnothing, MLD_monthly)
    MLD_stack = cat([dropdims(MLD_monthly[m]; dims=3) for m in avail]...; dims=3)
    MLD_min = dropdims(minimum(MLD_stack; dims=3); dims=3)
    MLD_max = dropdims(maximum(MLD_stack; dims=3); dims=3)

    # Sea-ice March / September
    SIC_mar = compute_monthly_mean(sic, [3]; start_time, stop_time)
    SIC_sep = compute_monthly_mean(sic, [9]; start_time, stop_time)
    SIC_mar = isnothing(SIC_mar) ? nothing : dropdims(SIC_mar; dims=3)
    SIC_sep = isnothing(SIC_sep) ? nothing : dropdims(SIC_sep; dims=3)

    # WOA comparison
    T_woa = Field(Metadatum(:temperature; dataset = WOAAnnual()), CPU())
    S_woa = Field(Metadatum(:salinity;    dataset = WOAAnnual()), CPU())
    T_interp = CenterField(grid); interpolate!(T_interp, T_woa)
    S_interp = CenterField(grid); interpolate!(S_interp, S_woa)
    T_woa_on_grid = Array(interior(T_interp))
    S_woa_on_grid = Array(interior(S_interp))
    δSST = SST .- T_woa_on_grid[:, :, Nz]
    δSSS = SSS .- S_woa_on_grid[:, :, Nz]

    # Apply land mask
    for f in (SST, SSS, SSH, HF, FW, SIC_mean, SSH_var, MLD_min, MLD_max, δSST, δSSS)
        mask_land!(f, land)
    end
    !isnothing(SIC_mar) && mask_land!(SIC_mar, land)
    !isnothing(SIC_sep) && mask_land!(SIC_sep, land)

    return (; grid, Nx, Ny, Nz, land, surface_file,
              SST, SSS, SSH, HF, FW, SIC_mean, SSH_var,
              MLD_min, MLD_max, SIC_mar, SIC_sep,
              δSST, δSSS, T_woa_on_grid, S_woa_on_grid)
end

D = Dict{String, Any}()
labels = [c.label for c in cases]
for c in cases
    @info "Loading surface: $(c.label)..."
    D[c.label] = load_surface_case(c.run_dir, c.prefix; start_time, stop_time)
end
@info "Done."

## Figure 1 -- SST bias vs WOA

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].δSST;
           title = "$lab: SST - WOA", colormap = :balance,
           colorrange = (-5, 5), label = "deg C")
end
fig

## Figure 2 -- SSS bias vs WOA

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].δSSS;
           title = "$lab: SSS - WOA", colormap = :balance,
           colorrange = (-3, 3), label = "PSU")
end
fig

## Figure 3 -- SSH

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].SSH;
           title = "$lab: Time-mean SSH", colormap = :balance,
           colorrange = (-2, 2), label = "m")
end
fig

## Figure 4 -- Min / Max monthly-mean MLD (cf. OM4 Figs. 3-4)

In [ ]:
fig = Figure(size = (800 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].MLD_min;
           title = "$lab: Min MLD (summer)",
           colormap = Reverse(:deep), colorrange = (0, 150), label = "m")
    panel!(fig, [2, 2i-1], D[lab].MLD_max;
           title = "$lab: Max MLD (winter)",
           colormap = Reverse(:deep), colorrange = (10, 3000), label = "m")
end
fig

## Figure 5 -- Sea-ice concentration March / September

In [ ]:
fig = Figure(size = (800 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    d = D[lab]
    if !isnothing(d.SIC_mar)
        panel!(fig, [1, 2i-1], d.SIC_mar;
               title = "$lab: Sea-ice conc. March",
               colormap = :ice, colorrange = (0, 1), label = "fraction")
    end
    if !isnothing(d.SIC_sep)
        panel!(fig, [2, 2i-1], d.SIC_sep;
               title = "$lab: Sea-ice conc. September",
               colormap = :ice, colorrange = (0, 1), label = "fraction")
    end
end
fig

## Figure 6 -- Surface fluxes

In [ ]:
fig = Figure(size = (800 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].HF;
           title = "$lab: Net heat flux", colormap = :balance,
           colorrange = (-200, 200), label = "W/m^2")
    panel!(fig, [2, 2i-1], D[lab].FW;
           title = "$lab: Net freshwater flux", colormap = :balance,
           colorrange = (-1e-4, 1e-4), label = "kg/m^2/s")
end
fig

## Figure 7 -- SSH variance

In [ ]:
fig = Figure(size = (800 * length(labels), 500), fontsize = 14)
for (i, lab) in enumerate(labels)
    panel!(fig, [1, 2i-1], D[lab].SSH_var;
           title = "$lab: SSH variance", colormap = :magma,
           colorrange = (0, 0.05), label = "m\u00b2")
end
fig

## Figure 8 -- Sea-ice volume: Arctic and Antarctic

Monthly climatology of sea-ice volume ( $\int h \, \alpha \, \mathrm{d}A$ ) split by hemisphere using `Integral` with `condition`.

In [ ]:
using Oceananigans.Grids: φnode

arctic_condition(i, j, k, grid, args...)    = φnode(i, j, k, grid, Center(), Center(), Center()) > 0
antarctic_condition(i, j, k, grid, args...) = φnode(i, j, k, grid, Center(), Center(), Center()) < 0

function compute_ice_diagnostics(run_dir, prefix, grid;
                                 start_time = 0, stop_time = Inf,
                                 reference_date = DateTime(1958, 1, 1),
                                 extent_threshold = 0.15)
    surface_file      = find_first_file(run_dir, prefix, "surface")
    thickness_fts     = FieldTimeSeries(surface_file, "sithick"; backend = OnDisk())
    concentration_fts = FieldTimeSeries(surface_file, "siconc";  backend = OnDisk())

    Nt = length(thickness_fts.times)
    arctic_volume      = zeros(Nt)
    antarctic_volume   = zeros(Nt)
    antarctic_extent   = zeros(Nt)
    snapshot_dates     = [reference_date + Second(round(Int, t)) for t in thickness_fts.times]

    # Persistent fields for extent computation
    extent_mask = Field{Center, Center, Nothing}(grid)
    antarctic_extent_integral = Field(Integral(extent_mask; condition = antarctic_condition))

    for n in 1:Nt
        # Volume: ∫ h ℵ dA per hemisphere
        ice_volume_field   = thickness_fts[n] * concentration_fts[n]
        arctic_integral    = Field(Integral(ice_volume_field; condition = arctic_condition))
        antarctic_integral = Field(Integral(ice_volume_field; condition = antarctic_condition))
        compute!(arctic_integral)
        compute!(antarctic_integral)
        arctic_volume[n]    = arctic_integral[1, 1, 1]
        antarctic_volume[n] = antarctic_integral[1, 1, 1]

        # Extent: ∫ dA where ℵ > threshold, southern hemisphere
        concentration_data = Array(interior(concentration_fts[n], :, :, 1))
        set!(extent_mask, Float64.(concentration_data .> extent_threshold))
        compute!(antarctic_extent_integral)
        antarctic_extent[n] = antarctic_extent_integral[1, 1, 1]
    end

    # Monthly climatology (only within [start_time, stop_time])
    idx = findall(t -> start_time <= t <= stop_time, thickness_fts.times)
    months_used = month.(snapshot_dates[idx])
    arctic_volume_monthly    = [mean(arctic_volume[idx[months_used .== m]])    for m in 1:12]
    antarctic_volume_monthly = [mean(antarctic_volume[idx[months_used .== m]]) for m in 1:12]
    antarctic_extent_monthly = [mean(antarctic_extent[idx[months_used .== m]]) for m in 1:12]

    return (; arctic_volume, antarctic_volume, antarctic_extent, snapshot_dates,
              arctic_volume_monthly, antarctic_volume_monthly, antarctic_extent_monthly)
end

ICE = Dict{String, Any}()
for c in cases
    @info "Computing sea-ice diagnostics for $(c.label)..."
    ICE[c.label] = compute_ice_diagnostics(c.run_dir, c.prefix, D[c.label].grid; start_time, stop_time)
end
@info "Done."

In [ ]:
# Download observational sea-ice climatologies
using Downloads, DelimitedFiles

# --- PIOMAS Arctic sea-ice volume (Schweiger et al., 2011) ---
# Units: 10^3 km^3
piomas_url  = "https://psc.apl.uw.edu/wordpress/wp-content/uploads/schweiger/ice_volume/PIOMAS.monthly.Current.v2.1.csv"
piomas_file = Downloads.download(piomas_url)
piomas_raw  = readdlm(piomas_file, ','; skipstart=1)
piomas_years  = Int.(piomas_raw[:, 1])
piomas_volume = Float64.(piomas_raw[:, 2:13])
piomas_volume[piomas_volume .== -1] .= NaN
piomas_monthly = vec(mapslices(x -> mean(filter(!isnan, x)), piomas_volume; dims=1))
@info "PIOMAS: $(piomas_years[1])-$(piomas_years[end])"

# --- NSIDC Sea Ice Index v4.0: Antarctic extent (Fetterer et al.) ---
# Units: million km^2
nsidc_antarctic_monthly = zeros(12)
for m in 1:12
    url = "https://noaadata.apps.nsidc.org/NOAA/G02135/south/monthly/data/S_$(lpad(m, 2, '0'))_extent_v4.0.csv"
    raw = readlines(Downloads.download(url))
    extents = Float64[]
    for line in raw
        parts = split(line, ',')
        length(parts) >= 5 || continue
        val = tryparse(Float64, strip(parts[5]))
        (isnothing(val) || val == -9999) && continue
        push!(extents, val)
    end
    nsidc_antarctic_monthly[m] = mean(extents)
end
@info "NSIDC Antarctic extent: 12-month climatology loaded"

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
m3_to_1e3km3  = 1e-12   # model m^3  -> 10^3 km^3  (PIOMAS units)
m2_to_Mkm2    = 1e-12   # model m^2  -> 10^6 km^2  (NSIDC units)

fig = Figure(size = (1200, 500), fontsize = 14)

# --- Arctic volume ---
ax = Axis(fig[1, 1]; xlabel = "Month", ylabel = "Ice volume (10\u00b3 km\u00b3)",
          title = "Arctic sea-ice volume",
          xticks = (1:12, month_names))
lines!(ax, 1:12, piomas_monthly;
       color = :black, linestyle = :dash, linewidth = 2, label = "PIOMAS")
for (i, lab) in enumerate(labels)
    lines!(ax, 1:12, ICE[lab].arctic_volume_monthly .* m3_to_1e3km3;
           color = case_colors[i], label = lab)
end
axislegend(ax; position = :rt)

# --- Antarctic extent ---
ax = Axis(fig[1, 2]; xlabel = "Month", ylabel = "Ice extent (10\u2076 km\u00b2)",
          title = "Antarctic sea-ice extent",
          xticks = (1:12, month_names))
lines!(ax, 1:12, nsidc_antarctic_monthly;
       color = :black, linestyle = :dash, linewidth = 2, label = "NSIDC")
for (i, lab) in enumerate(labels)
    lines!(ax, 1:12, ICE[lab].antarctic_extent_monthly .* m2_to_Mkm2;
           color = case_colors[i], label = lab)
end
axislegend(ax; position = :rt)

fig

function load_timeseries_case(run_dir, prefix, grid; start_time = 0, stop_time = Inf)
    # Global-mean drift (full time series -- drift is always from t=0)
    averages_file = find_first_file(run_dir, prefix, "averages")
    temperature_mean_fts = FieldTimeSeries(averages_file, "tosga"; backend = OnDisk())
    salinity_mean_fts    = FieldTimeSeries(averages_file, "soga";  backend = OnDisk())
    temperature_mean = [Array(interior(temperature_mean_fts[n]))[1] for n in 1:length(temperature_mean_fts.times)]
    salinity_mean    = [Array(interior(salinity_mean_fts[n]))[1]  for n in 1:length(salinity_mean_fts.times)]
    time_in_years    = temperature_mean_fts.times ./ (365.25 * 24 * 3600)

    # Horizontal-mean profiles (averaged within window)
    temperature_profile_fts = FieldTimeSeries(averages_file, "to_h"; backend = OnDisk())
    salinity_profile_fts    = FieldTimeSeries(averages_file, "so_h"; backend = OnDisk())
    temperature_profile = vec(compute_time_mean(temperature_profile_fts; start_time, stop_time))
    salinity_profile    = vec(compute_time_mean(salinity_profile_fts; start_time, stop_time))
    depth = collect(znodes(grid, Center()))

    # Global-mean TKE (full time series)
    fields_file = find_first_file(run_dir, prefix, "fields")
    tke_fts     = FieldTimeSeries(fields_file, "tke"; backend = OnDisk())
    ocean_mask  = build_ocean_mask_3d(grid)
    ocean_cells = sum(ocean_mask)
    tke_mean = [sum(Array(interior(tke_fts[n])) .* ocean_mask) / ocean_cells
                for n in 1:length(tke_fts.times)]
    tke_time_in_years = tke_fts.times ./ (365.25 * 24 * 3600)

    return (; temperature_mean, salinity_mean, time_in_years,
              temperature_profile, salinity_profile, depth,
              tke_mean, tke_time_in_years, ocean_mask, fields_file)
end

TS = Dict{String, Any}()
for c in cases
    @info "Loading time series: $(c.label)..."
    TS[c.label] = load_timeseries_case(c.run_dir, c.prefix, D[c.label].grid; start_time, stop_time)
end
@info "Done."

In [ ]:
function load_timeseries_case(run_dir, prefix, grid; start_time = 0)
    # Global-mean drift (full time series -- drift is always from t=0)
    averages_file = find_first_file(run_dir, prefix, "averages")
    temperature_mean_fts = FieldTimeSeries(averages_file, "tosga"; backend = OnDisk())
    salinity_mean_fts    = FieldTimeSeries(averages_file, "soga";  backend = OnDisk())
    temperature_mean = [Array(interior(temperature_mean_fts[n]))[1] for n in 1:length(temperature_mean_fts.times)]
    salinity_mean    = [Array(interior(salinity_mean_fts[n]))[1]  for n in 1:length(salinity_mean_fts.times)]
    time_in_years    = temperature_mean_fts.times ./ (365.25 * 24 * 3600)

    # Horizontal-mean profiles (averaged from start_time)
    temperature_profile_fts = FieldTimeSeries(averages_file, "to_h"; backend = OnDisk())
    salinity_profile_fts    = FieldTimeSeries(averages_file, "so_h"; backend = OnDisk())
    temperature_profile = vec(compute_time_mean(temperature_profile_fts; start_time))
    salinity_profile    = vec(compute_time_mean(salinity_profile_fts; start_time))
    depth = collect(znodes(grid, Center()))

    # Global-mean TKE (full time series)
    fields_file = find_first_file(run_dir, prefix, "fields")
    tke_fts     = FieldTimeSeries(fields_file, "tke"; backend = OnDisk())
    ocean_mask  = build_ocean_mask_3d(grid)
    ocean_cells = sum(ocean_mask)
    tke_mean = [sum(Array(interior(tke_fts[n])) .* ocean_mask) / ocean_cells
                for n in 1:length(tke_fts.times)]
    tke_time_in_years = tke_fts.times ./ (365.25 * 24 * 3600)

    return (; temperature_mean, salinity_mean, time_in_years,
              temperature_profile, salinity_profile, depth,
              tke_mean, tke_time_in_years, ocean_mask, fields_file)
end

TS = Dict{String, Any}()
for c in cases
    @info "Loading time series: $(c.label)..."
    TS[c.label] = load_timeseries_case(c.run_dir, c.prefix, D[c.label].grid; start_time)
end
@info "Done."

## Figure 9 -- Global-mean TKE

In [ ]:
fig = Figure(size = (900, 450), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Time (years)", ylabel = "TKE (m\u00b2/s\u00b2)",
          title = "Global-mean turbulent kinetic energy")
for (i, lab) in enumerate(labels)
    lines!(ax, TS[lab].tke_time_in_years, TS[lab].tke_mean;
           color = case_colors[i], label = lab)
end
axislegend(ax; position = :rb)
fig

## Figure 10 -- Global-mean T and S drift

In [ ]:
fig = Figure(size = (1200, 450), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Time (years)", ylabel = "\u0394T (deg C)",
          title = "Global-mean temperature drift")
for (i, lab) in enumerate(labels)
    d = TS[lab]
    lines!(ax, d.time_in_years, d.temperature_mean .- d.temperature_mean[1];
           color = case_colors[i], label = lab)
end
axislegend(ax; position = :lb)

ax = Axis(fig[1, 2]; xlabel = "Time (years)", ylabel = "\u0394S (PSU)",
          title = "Global-mean salinity drift")
for (i, lab) in enumerate(labels)
    d = TS[lab]
    lines!(ax, d.time_in_years, d.salinity_mean .- d.salinity_mean[1];
           color = case_colors[i], label = lab)
end
axislegend(ax; position = :lb)
fig

## Figure 11 -- Horizontal-mean T and S profiles

In [ ]:
fig = Figure(size = (1000, 600), fontsize = 14)
ax = Axis(fig[1, 1]; xlabel = "Temperature (deg C)", ylabel = "Depth (m)",
          title = "Horizontal-mean temperature")
for (i, lab) in enumerate(labels)
    lines!(ax, TS[lab].temperature_profile, TS[lab].depth;
           color = case_colors[i], label = lab)
end
ylims!(ax, (-5500, 0)); axislegend(ax; position = :rb)

ax = Axis(fig[1, 2]; xlabel = "Salinity (PSU)", ylabel = "Depth (m)",
          title = "Horizontal-mean salinity")
for (i, lab) in enumerate(labels)
    lines!(ax, TS[lab].salinity_profile, TS[lab].depth;
           color = case_colors[i], label = lab)
end
ylims!(ax, (-5500, 0)); axislegend(ax; position = :rb)
fig

---
## Zonal-mean sections

Regrid 3-D time-mean fields to a regular 1-degree lat-lon grid via
`ConservativeRegridding`, then average over longitude.
A per-case regridder is built (this is expensive).

In [ ]:
ZM = Dict{String, Any}()

for c in cases
    lab  = c.label
    grid = D[lab].grid
    ocean_mask = TS[lab].ocean_mask

    # Build per-case regridder
    @info "Building regridder for $lab (may take a few minutes)..."
    src_f = Field{Center, Center, Nothing}(grid)
    regridder = ConservativeRegridding.Regridder(dst_f, src_f; progress = true)

    # Load 3-D fields
    @info "Loading 3-D fields for $lab..."
    fields_file = TS[lab].fields_file
    to_fts = FieldTimeSeries(fields_file, "to"; backend = OnDisk())
    so_fts = FieldTimeSeries(fields_file, "so"; backend = OnDisk())
    bo_fts = FieldTimeSeries(fields_file, "bo"; backend = OnDisk())

    temperature_mean = compute_time_mean(to_fts; start_time, stop_time)
    salinity_mean    = compute_time_mean(so_fts; start_time, stop_time)
    buoyancy_mean    = compute_time_mean(bo_fts; start_time, stop_time)
    buoyancy_initial = Array(interior(bo_fts[1]))

    @info "Computing zonal means for $lab..."
    temperature_zonal = compute_zonal_mean(temperature_mean, ocean_mask, regridder, Nlon, Nlat)
    salinity_zonal    = compute_zonal_mean(salinity_mean,    ocean_mask, regridder, Nlon, Nlat)
    buoyancy_zonal    = compute_zonal_mean(buoyancy_mean,    ocean_mask, regridder, Nlon, Nlat)

    temperature_woa_zonal = compute_zonal_mean(D[lab].T_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
    salinity_woa_zonal    = compute_zonal_mean(D[lab].S_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
    buoyancy_init_zonal   = compute_zonal_mean(buoyancy_initial,     ocean_mask, regridder, Nlon, Nlat)

    depth = collect(znodes(grid, Center()))

    ZM[lab] = (; temperature_zonal, salinity_zonal, buoyancy_zonal,
                δtemperature_zonal = temperature_zonal .- temperature_woa_zonal,
                δsalinity_zonal    = salinity_zonal    .- salinity_woa_zonal,
                δbuoyancy_zonal    = buoyancy_zonal    .- buoyancy_init_zonal,
                depth)
end

latitude = collect(φnodes(latlon_grid, Center()))
@info "Zonal means done."

In [ ]:
ZM = Dict{String, Any}()

for c in cases
    lab  = c.label
    grid = D[lab].grid
    ocean_mask = TS[lab].ocean_mask

    # Build per-case regridder
    @info "Building regridder for $lab (may take a few minutes)..."
    src_f = Field{Center, Center, Nothing}(grid)
    regridder = ConservativeRegridding.Regridder(dst_f, src_f; progress = true)

    # Load 3-D fields
    @info "Loading 3-D fields for $lab..."
    fields_file = TS[lab].fields_file
    to_fts = FieldTimeSeries(fields_file, "to"; backend = OnDisk())
    so_fts = FieldTimeSeries(fields_file, "so"; backend = OnDisk())
    bo_fts = FieldTimeSeries(fields_file, "bo"; backend = OnDisk())

    temperature_mean = compute_time_mean(to_fts; start_time)
    salinity_mean    = compute_time_mean(so_fts; start_time)
    buoyancy_mean    = compute_time_mean(bo_fts; start_time)
    buoyancy_initial = Array(interior(bo_fts[1]))

    @info "Computing zonal means for $lab..."
    temperature_zonal = compute_zonal_mean(temperature_mean, ocean_mask, regridder, Nlon, Nlat)
    salinity_zonal    = compute_zonal_mean(salinity_mean,    ocean_mask, regridder, Nlon, Nlat)
    buoyancy_zonal    = compute_zonal_mean(buoyancy_mean,    ocean_mask, regridder, Nlon, Nlat)

    temperature_woa_zonal = compute_zonal_mean(D[lab].T_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
    salinity_woa_zonal    = compute_zonal_mean(D[lab].S_woa_on_grid, ocean_mask, regridder, Nlon, Nlat)
    buoyancy_init_zonal   = compute_zonal_mean(buoyancy_initial,     ocean_mask, regridder, Nlon, Nlat)

    depth = collect(znodes(grid, Center()))

    ZM[lab] = (; temperature_zonal, salinity_zonal, buoyancy_zonal,
                δtemperature_zonal = temperature_zonal .- temperature_woa_zonal,
                δsalinity_zonal    = salinity_zonal    .- salinity_woa_zonal,
                δbuoyancy_zonal    = buoyancy_zonal    .- buoyancy_init_zonal,
                depth)
end

latitude = collect(φnodes(latlon_grid, Center()))
@info "Zonal means done."

## Figure 12 -- Zonal-mean T, S, b

In [ ]:
fig = Figure(size = (600 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    zm = ZM[lab]
    ax = Axis(fig[1, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal T")
    hm = heatmap!(ax, latitude, zm.depth, zm.temperature_zonal; colormap=:thermal,
                  colorrange=(-2,30), nan_color=:lightgray)
    Colorbar(fig[1, 2i], hm; label="deg C")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[2, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal S")
    hm = heatmap!(ax, latitude, zm.depth, zm.salinity_zonal; colormap=:haline,
                  colorrange=(33,37), nan_color=:lightgray)
    Colorbar(fig[2, 2i], hm; label="PSU")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[3, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal b")
    hm = heatmap!(ax, latitude, zm.depth, zm.buoyancy_zonal; colormap=:balance,
                  nan_color=:lightgray)
    Colorbar(fig[3, 2i], hm; label="m/s\u00b2")
    ylims!(ax, (-5500, 0))
end
fig

## Figure 13 -- Zonal-mean drift from initial conditions

In [ ]:
fig = Figure(size = (600 * length(labels), 900), fontsize = 14)
for (i, lab) in enumerate(labels)
    zm = ZM[lab]
    ax = Axis(fig[1, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal T - WOA")
    hm = heatmap!(ax, latitude, zm.depth, zm.δtemperature_zonal; colormap=:balance,
                  colorrange=(-5,5), nan_color=:lightgray)
    Colorbar(fig[1, 2i], hm; label="deg C")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[2, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal S - WOA")
    hm = heatmap!(ax, latitude, zm.depth, zm.δsalinity_zonal; colormap=:balance,
                  colorrange=(-1,1), nan_color=:lightgray)
    Colorbar(fig[2, 2i], hm; label="PSU")
    ylims!(ax, (-5500, 0))

    ax = Axis(fig[3, 2i-1]; xlabel="Latitude", ylabel="Depth (m)",
              title="$lab: Zonal b - b(t=0)")
    hm = heatmap!(ax, latitude, zm.depth, zm.δbuoyancy_zonal; colormap=:balance,
                  nan_color=:lightgray)
    Colorbar(fig[3, 2i], hm; label="m/s\u00b2")
    ylims!(ax, (-5500, 0))
end
fig